In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import numpy as np
from collections import Counter
from generator import make_scenario
from env import play, score
from bots import random_bots

G = yaml.safe_load(open("config.yaml"))["calibration"]["g_points"]
print(f"g = {G}")


In [ ]:
def fuzz(seed=12345, n=1000):
    rng = np.random.default_rng(seed)
    quote, respond, decide = random_bots(rng)

    proposals = []
    def recording_respond(cv, g, p):
        action, q = respond(cv, g, p)
        proposals.append((action, q, p))
        return action, q

    scns, eps, outs = [], [], []
    for s in range(1, n + 1):
        scn = make_scenario(s, G)
        ep = play(scn, G, quote, recording_respond, decide)
        scns.append(scn); eps.append(ep); outs.append(score(scn, ep))
    return scns, eps, outs, proposals

scns, eps, outs, proposals = fuzz()
print(f"{len(eps)} episodes")

In [ ]:
for scn, out in zip(scns, outs):
    assert (out.price is None) == (out.status == "counter_rejected"), out.seed
    assert out.no_deal == (1 if out.price is None else 0), out.seed
    if out.price is not None:
        S = scn.r - scn.c
        assert abs(out.dealer_surplus + out.client_surplus - S) < 1e-9, out.seed
print("(a) consistency OK")


In [ ]:
rewrites = 0
for (action, q, p), ep in zip(proposals, eps):
    if action == "counter" and q >= p:
        rewrites += 1
        assert ep.status == "quote_accepted", ep.seed
        assert ep.counter is None, ep.seed
        assert ep.price == p, ep.seed
assert rewrites > 0, "rewrite branch never exercised"
print(f"(b) rewrite fired {rewrites} times, all handled correctly")


In [ ]:
status = Counter(o.status for o in outs)
viol   = Counter(v for o in outs for v in o.violations)

VIOLS = ("dealer_quote_below_cost", "client_counter_above_limit",
         "client_accepted_above_limit", "dealer_rejected_profitable_counter")

print("statuses:")
for k, n in status.most_common():
    print(f"  {k:<22}{n:>5}")
print("\nviolations:")
for name in VIOLS:
    print(f"  {name:<38}{viol[name]:>5}")

assert len(status) == 3, f"only reached {sorted(status)}"
for name in VIOLS:
    assert viol[name] > 0, f"{name} never triggered"
print("\n(c) all 3 statuses and all 4 violations reached")


In [ ]:
assert fuzz(12345)[1] == fuzz(12345)[1]
print("(d) replay OK")
